# 11. 한국어 일기 감정 분석 — 최종 사용자 실행 버전

**작성자: 김민서**

기존 `10_diary_system.ipynb`의 **KoBERT 다중 감정 분석, 감정별 threshold, 문맥 검증, 7개 대표 감정, 감정 달력, 긍정·부정 그래프 로직은 그대로 유지**합니다.

11에서는 NLP 분석 알고리즘을 새로 바꾸지 않고, 사용자가 코드를 직접 수정하지 않아도 사용할 수 있도록 사용자 화면과 데이터 흐름을 정리했습니다.

### 최종 수정 사항
1. **테스트 일기 문장 수 정정**
   - 30개 일기의 실제 `split_sentences()` 결과가 2~5문장으로 정확히 분포합니다.
   - 분포: 2문장 7개 / 3문장 8개 / 4문장 8개 / 5문장 7개
2. **중복 초기 분석 제거**
   - 10의 기존 `sample_diary.csv`를 먼저 분석하지 않습니다.
   - 선택된 데이터셋만 한 번 분석합니다.
3. **11 전용 결과 파일 사용**
   - 10의 `diary_daily_context_v4.csv`, `diary_sentences_context_v4.csv`를 덮어쓰지 않습니다.
4. **데모/실사용 데이터 분리**
   - `데모 데이터 30개`: 2026년 9월 기준 테스트 데이터, 저장 비활성화
   - `내 일기`: 사용자가 직접 저장한 일기만 별도 파일에 저장

### 사용 방법
1. 프로젝트의 `.venv` 커널을 선택합니다.
2. 이 노트북에서 **Run All**을 실행합니다.
3. 상단에서 `데모 데이터 30개` 또는 `내 일기`를 선택합니다.
4. 아래 세 탭을 사용합니다.
   - **일기 작성**: 날짜 선택 → 일기 작성 → 감정 분석 / 분석 후 저장
   - **월간 감정**: 연도·월 선택 → 감정 요약·달력·긍정/부정 그래프
   - **날짜별 조회**: 저장된 날짜 선택 → 원문 일기와 분석 결과 확인

5. **초기화 순서 오류 수정**
   - `daily`가 생성되기 전에 10의 월간 셀이 실행되던 문제를 수정했습니다.
   - 그래프/월간 관련 셀은 함수 정의만 가져오고, 실제 출력은 사용자 버튼을 눌렀을 때만 수행합니다.

6. **빈 데이터 월간 조회 처리**
   - `내 일기` 모드에 저장된 일기가 0개여도 오류가 발생하지 않습니다.
   - 월간 조회 시 `"아직 저장된 일기가 없습니다"` 안내를 표시합니다.


In [1]:
# 1. 사용자용 시스템 초기화
# 10번 노트북의 NLP 분석 로직만 가져오고,
# 10의 sample_diary 분석/테스트 출력은 실행하지 않습니다.

import json
import sys
from pathlib import Path
from collections import Counter
from IPython.utils.capture import capture_output

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError(
        "사용자 화면에 ipywidgets가 필요합니다.\n"
        "프로젝트 .venv 터미널에서 다음 명령을 한 번 실행하세요:\n"
        "python -m pip install ipywidgets"
    ) from exc

cwd = Path.cwd()
notebooks_dir = cwd if cwd.name == "notebooks" else cwd / "notebooks"
project_root_11 = notebooks_dir.parent
base_notebook = notebooks_dir / "10_diary_system.ipynb"
model_file_11 = project_root_11 / "models" / "kobert_kote_nall_e5.pt"

if not base_notebook.exists():
    raise FileNotFoundError(
        f"10번 노트북을 찾을 수 없습니다:\n{base_notebook}\n\n"
        "11 파일과 10_diary_system.ipynb를 같은 notebooks 폴더에 두세요."
    )

if not model_file_11.exists():
    raise FileNotFoundError(
        f"KoBERT 모델 파일을 찾을 수 없습니다:\n{model_file_11}\n\n"
        "프로젝트의 models 폴더에 kobert_kote_nall_e5.pt를 넣어주세요."
    )

expected_venv = str((project_root_11 / ".venv").resolve()).lower()
current_python = str(Path(sys.executable).resolve()).lower()

print("감정 분석 시스템을 준비하고 있습니다.")
print("처음 실행하거나 calibration 캐시가 없으면 준비 시간이 더 걸릴 수 있습니다.")

if expected_venv not in current_python:
    print("\n[커널 안내]")
    print("현재:", sys.executable)
    print("권장:", project_root_11 / ".venv" / "Scripts" / "python.exe")

nb10 = json.loads(base_notebook.read_text(encoding="utf-8"))

# 10의 NLP 핵심 함수/화면 함수만 가져옵니다.
# 제외:
# - cell 13: 기존 sample_diary.csv 로드
# - cell 15의 즉시 전체 분석 호출
# - 개발용 비교/회귀/긴 일기/데모 셀
# 10의 NLP 핵심 셀 중 "정의"가 필요한 부분만 실행합니다.
# daily/SELECT_YEAR가 필요한 즉시 실행 코드는 제외합니다.
CORE_CELLS = [
    1,   # 기본 설정
    2,   # 설정
    3,   # KOTE 44개 감정 + 그룹
    4,   # KoBERT 모델 로드
    5,   # 예측 캐시
    6,   # 감정별 threshold calibration
    7,   # 문맥 검증
    9,   # 문장 분리
    10,  # 그룹 강도 / 대표 감정
    11,  # analyze_text
    14,  # 하루 코멘트
    18,  # 감정 달력 공통 함수 정의
    27,  # 카드 스타일/도우미
    28,  # 하루 카드 + 저장 함수 정의
    29,  # 월간 카드 함수 정의
]

with capture_output():
    for cell_index in CORE_CELLS:
        source = "".join(nb10["cells"][cell_index].get("source", []))
        exec(
            compile(
                source,
                f"10_diary_system.ipynb:cell_{cell_index}",
                "exec"
            ),
            globals()
        )

# cell 15: build_daily_outputs() 함수 정의만 가져옵니다.
# 기존 sample_diary.csv를 즉시 분석하는 코드는 실행하지 않습니다.
cell15_source = "".join(nb10["cells"][15].get("source", []))
cell15_function_only = cell15_source.split(
    "daily, sentences = build_daily_outputs()"
)[0]
with capture_output():
    exec(
        compile(
            cell15_function_only,
            "10_diary_system.ipynb:cell_15_function_only",
            "exec"
        ),
        globals()
    )

# cell 17: get_month_data() 함수만 가져옵니다.
# 원본 셀 뒤쪽의 `last_date = daily[...]` 부분은 daily 생성 전이므로 실행하지 않습니다.
cell17_source = "".join(nb10["cells"][17].get("source", []))
cell17_function_only = cell17_source.split(
    'last_date = daily["date"].max()'
)[0]
with capture_output():
    exec(
        compile(
            cell17_function_only,
            "10_diary_system.ipynb:cell_17_function_only",
            "exec"
        ),
        globals()
    )

# cell 19: 그래프 함수 정의만 가져옵니다.
# 원본 셀의 즉시 그래프 출력은 사용자 화면에서 버튼을 눌렀을 때만 실행합니다.
cell19_source = "".join(nb10["cells"][19].get("source", []))
cell19_function_only = cell19_source.split(
    "plot_positive_graph(SELECT_YEAR, SELECT_MONTH)"
)[0]
with capture_output():
    exec(
        compile(
            cell19_function_only,
            "10_diary_system.ipynb:cell_19_function_only",
            "exec"
        ),
        globals()
    )

# cell 20: 감정 달력 함수 정의만 가져옵니다.
cell20_source = "".join(nb10["cells"][20].get("source", []))
cell20_function_only = cell20_source.split(
    "plot_emotion_calendar(SELECT_YEAR, SELECT_MONTH)"
)[0]
with capture_output():
    exec(
        compile(
            cell20_function_only,
            "10_diary_system.ipynb:cell_20_function_only",
            "exec"
        ),
        globals()
    )


# ---------------------------------------------------------
# 11 전용 데이터 모드
# ---------------------------------------------------------
DEMO_MODE_DEFAULT = True
DEMO_YEAR = 2026
DEMO_MONTH = 9

DEMO_DIARY_FILE = data_dir / "sample_diary_11_demo_v2.csv"
DEMO_TARGET_FILE = data_dir / "sample_diary_11_demo_targets_v2.csv"
USER_DIARY_FILE = data_dir / "sample_diary_11_user.csv"

CURRENT_DATA_MODE = "demo" if DEMO_MODE_DEFAULT else "user"
DIARY_FILE = DEMO_DIARY_FILE

EMBEDDED_TEST_DIARIES = [{'date': '2026-09-01', 'text': '조별 과제를 하는데 한 팀원이 맡은 일을 거의 준비하지 않았다. 결국 다른 사람들이 대신 해야 하는 상황이 생겼다. 약속한 역할을 지키지 않은 것이 너무 답답했다. 최소한 자기 몫은 책임졌으면 좋겠다는 생각에 화가 났다. 한동안 불편한 마음이 가라앉지 않았다.', 'target_emotion': '분노', 'sentence_count': 5}, {'date': '2026-09-02', 'text': '오늘 새로운 프로젝트 주제를 정하고 앞으로 구현할 기능도 하나씩 정리했다. 아직 시작 단계라 직접 만들어 볼 생각에 기대되고 결과가 어떻게 나올지 궁금해서 빨리 작업을 시작하고 싶다.', 'target_emotion': '기대', 'sentence_count': 2}, {'date': '2026-09-03', 'text': '오늘 친하게 지내던 친구가 다른 지역으로 이사를 간다는 이야기를 들었다. 앞으로 자주 볼 수 없다는 생각에 마음이 많이 허전했다. 집에 돌아와서도 함께했던 일들이 계속 생각났다. 평소처럼 쉽게 만나지 못할 것 같아 슬펐다.', 'target_emotion': '슬픔', 'sentence_count': 4}, {'date': '2026-09-04', 'text': '이번 주 내내 시험과 과제가 이어져 제대로 쉬지 못했다. 해야 할 일은 아직 남아 있다. 집중력이 떨어져서 더 이상 머리가 돌아가지 않는 느낌이다. 몸도 마음도 지쳐서 잠시 아무것도 하지 않고 쉬고 싶다.', 'target_emotion': '피로', 'sentence_count': 4}, {'date': '2026-09-05', 'text': '오늘은 정해진 시간에 수업에 참석했다. 수업이 끝난 뒤 집으로 와서 간단히 저녁을 먹었다. 이후에는 다음 날 필요한 파일을 정리하고 하루를 마무리했다. 이후에는 평소처럼 남은 일정을 정리했다.', 'target_emotion': '중립', 'sentence_count': 4}, {'date': '2026-09-06', 'text': '오늘 약속 시간에 친구가 아무 연락 없이 한참 늦게 와서 걱정하며 기다렸다. 별다른 이유도 없이 늦었다는 말을 듣자 미리 연락이라도 해줬으면 좋았을 텐데 화가 났다. 생각할수록 짜증이 났다.', 'target_emotion': '분노', 'sentence_count': 3}, {'date': '2026-09-07', 'text': '오늘 아침에 학교에 가서 오전 수업을 들었다. 점심을 먹은 뒤 도서관에서 필요한 자료를 찾아 정리했다. 저녁에는 집에 돌아와 내일 사용할 물건을 준비했다. 이후에는 평소처럼 남은 일정을 정리했다.', 'target_emotion': '중립', 'sentence_count': 4}, {'date': '2026-09-08', 'text': '수업 중에 계속 큰 소리로 이야기하는 사람들이 있어서 집중하기 어려웠다. 몇 번이나 조용해졌다가 다시 시끄러워져 중요한 내용을 놓친 것 같아 답답했다. 계속 방해받아서 결국 화가 났다.', 'target_emotion': '분노', 'sentence_count': 3}, {'date': '2026-09-09', 'text': '오늘 운동을 시작하면서 한 달 동안 지킬 목표를 세웠다. 지금은 아직 힘든 편이라 천천히 꾸준히 해보려고 한다. 계속하면 체력이 좋아질 것 같아 기대된다. 한 달 뒤에 얼마나 달라져 있을지 궁금하다. 앞으로의 변화가 더 기대된다.', 'target_emotion': '기대', 'sentence_count': 5}, {'date': '2026-09-10', 'text': '이번 달에 해야 할 과제와 시험 일정이 한꺼번에 몰려 있다. 제대로 다 끝낼 수 있을지 생각하면 벌써부터 걱정된다. 계획은 세워 놓았으나 시간이 부족할 것 같아 마음이 조급하다. 해야 할 일을 생각할수록 긴장된다.', 'target_emotion': '불안', 'sentence_count': 4}, {'date': '2026-09-11', 'text': '다음 주에 친구들과 오랜만에 여행을 가기로 했다. 어디를 갈지 찾아보는 것만으로도 벌써 설렌다. 좋은 추억을 많이 만들 수 있을 것 같아 기대된다. 여행 날짜가 빨리 왔으면 좋겠다.', 'target_emotion': '기대', 'sentence_count': 4}, {'date': '2026-09-12', 'text': '오늘은 평소와 비슷한 시간에 일어나 준비한 뒤 학교에 가서 수업을 들었다. 수업이 끝난 뒤 편의점에서 필요한 물건을 사고 집에 와서 저녁을 먹은 뒤 방을 정리했다.', 'target_emotion': '중립', 'sentence_count': 2}, {'date': '2026-09-13', 'text': '주말이라 늦잠을 자고 여유롭게 아침을 먹었다. 오후에는 좋아하는 음악을 들으며 산책했고 날씨까지 좋아서 기분이 상쾌했다. 특별한 일은 없었어도 마음이 편안했다. 하루를 돌아보니 행복한 하루였다는 생각이 들었다.', 'target_emotion': '행복', 'sentence_count': 4}, {'date': '2026-09-14', 'text': '오늘 친구와 오랜만에 만나 맛있는 점심을 먹고 별것 아닌 이야기를 하며 계속 웃어서 기분이 정말 좋았다. 집에 돌아오는 길에도 즐거웠던 순간이 생각나 오랜만에 마음 편하게 웃을 수 있어서 행복했다.', 'target_emotion': '행복', 'sentence_count': 2}, {'date': '2026-09-15', 'text': '며칠 동안 해결하지 못했던 코드 오류를 드디어 해결했다. 정상적으로 실행되는 화면을 보니 정말 기뻤다. 왜 문제가 생겼는지도 이해하게 됐다. 스스로 해결했다는 생각에 뿌듯하고 만족스러웠다. 하루를 돌아보니 좋은 기억이 많이 남았다.', 'target_emotion': '행복', 'sentence_count': 5}, {'date': '2026-09-16', 'text': '오늘 기대했던 약속이 갑자기 취소되어 오랜만에 만나려고 기다렸던 만큼 아쉬움이 컸다. 혼자 집에 돌아오니 괜히 마음이 허전했다. 기분이 가라앉아서 한동안 아무것도 하고 싶지 않았다.', 'target_emotion': '슬픔', 'sentence_count': 3}, {'date': '2026-09-17', 'text': '오늘 기다리던 과제 결과에서 좋은 점수를 받아 열심히 준비한 만큼 결과가 나온 것 같아 정말 뿌듯했다. 가족에게 소식을 전하니 같이 기뻐해 줘서 저녁까지 기분 좋은 마음이 계속됐다.', 'target_emotion': '행복', 'sentence_count': 2}, {'date': '2026-09-18', 'text': '집에 오는 길에 휴대폰 배터리가 거의 다 떨어졌는데 중요한 연락을 기다리고 있었다. 휴대폰이 꺼질까 봐 계속 불안했다. 충전할 곳을 찾을 때까지 마음을 놓을 수가 없었다.', 'target_emotion': '불안', 'sentence_count': 3}, {'date': '2026-09-19', 'text': '오전에는 온라인 강의를 듣고 내용을 노트에 정리했다. 오후에는 빨래를 하고 책상 위에 쌓여 있던 물건들을 정리했다. 저녁에는 내일 일정과 해야 할 일을 확인했다.', 'target_emotion': '중립', 'sentence_count': 3}, {'date': '2026-09-20', 'text': '오늘 주문한 음식이 한참 늦게 왔는데 주문 내용까지 잘못되어 있었다. 같은 설명을 다시 해야 해서 점점 짜증이 났다. 결국 다시 요청해야 해서 기분이 좋지 않았다.', 'target_emotion': '분노', 'sentence_count': 3}, {'date': '2026-09-21', 'text': '이번 주말에는 가족들과 오랜만에 외식을 하기로 했고 평소 가 보고 싶었던 식당에 갈 예정이다. 맛있는 음식을 먹으면서 편하게 이야기할 생각을 하니 기대돼서 주말이 빨리 왔으면 좋겠다.', 'target_emotion': '기대', 'sentence_count': 2}, {'date': '2026-09-22', 'text': '오랫동안 사용하던 물건을 실수로 잃어버렸다. 추억이 많이 담긴 물건이라 더 속상했다. 아무리 찾아도 보이지 않았다. 하루 종일 마음이 허전하고 슬펐다. 쉽게 기분이 나아지지는 않았다.', 'target_emotion': '슬픔', 'sentence_count': 5}, {'date': '2026-09-23', 'text': '오늘 시험을 봤는데 헷갈리는 문제가 생각보다 많았다. 시험이 끝난 뒤에도 내가 고른 답이 맞는지 계속 신경 쓰였다. 결과가 어떻게 나올지 걱정된다. 점수가 나올 때까지 마음이 편하지 않을 것 같다. 계속 생각이 나서 쉽게 마음을 놓기 어려웠다.', 'target_emotion': '불안', 'sentence_count': 5}, {'date': '2026-09-24', 'text': '열심히 준비했던 시험에서 생각보다 낮은 점수를 받았다. 노력한 만큼 결과가 나오지 않은 것 같아 많이 실망했다. 틀린 문제를 다시 보니 아는 내용도 실수했다. 아쉬운 마음이 계속 남아서 기분이 가라앉았다. 쉽게 기분이 나아지지는 않았다.', 'target_emotion': '슬픔', 'sentence_count': 5}, {'date': '2026-09-25', 'text': '오늘 친구가 내가 좋아하는 간식을 사다 줬다. 작은 일이었어도 나를 생각해 준 마음이 고마웠다. 함께 간식을 먹으며 한참 웃었다. 기분 좋게 하루를 마무리했다. 하루를 돌아보니 좋은 기억이 많이 남았다.', 'target_emotion': '행복', 'sentence_count': 5}, {'date': '2026-09-26', 'text': '오늘 하루 종일 밖에서 여러 일을 처리하느라 계속 돌아다녀 오후가 되면서 다리가 아프고 기운이 빠졌다. 집에 돌아오는 길에도 몸이 무거워서 도착하자마자 누워 있고 싶을 정도로 지쳤다.', 'target_emotion': '피로', 'sentence_count': 2}, {'date': '2026-09-27', 'text': '오늘 아침 일찍 학교에 가서 저녁까지 계속 수업을 들었고 중간에 쉴 시간이 거의 없었다. 오후부터 집중하기가 힘들었다. 집에 도착하니 아무것도 하고 싶지 않을 만큼 너무 피곤했다.', 'target_emotion': '피로', 'sentence_count': 3}, {'date': '2026-09-28', 'text': '다음 학기에 듣고 싶었던 수업의 신청 기간이 곧 시작된다. 평소 관심 있던 내용을 배울 수 있는 수업이다. 새로운 내용을 많이 배울 수 있을 것 같아 기대된다. 수강 신청에 꼭 성공하고 싶다.', 'target_emotion': '기대', 'sentence_count': 4}, {'date': '2026-09-29', 'text': '어제 늦게까지 과제를 하느라 잠을 거의 자지 못해서 아침부터 계속 졸리고 몸이 무거웠다. 수업 내용도 잘 들어오지 않았다. 오늘은 다른 일을 미루더라도 일찍 자고 푹 쉬고 싶다.', 'target_emotion': '피로', 'sentence_count': 3}, {'date': '2026-09-30', 'text': '내일 중요한 발표가 있는데 준비가 완벽하지 않은 것 같고 사람들 앞에서 말하다가 내용을 잊어버릴까 봐 걱정된다. 발표 생각만 하면 긴장돼서 연습을 더 해야 마음이 조금 놓일 것 같다.', 'target_emotion': '불안', 'sentence_count': 2}]

# 실제 split_sentences() 기준으로 문장 수가 정확한지 실행 시에도 검증합니다.
for item in EMBEDDED_TEST_DIARIES:
    actual_count = len(split_sentences(item["text"]))
    assert actual_count == item["sentence_count"], (
        f"문장 수 불일치: {item['date']} "
        f"저장={item['sentence_count']}, 실제={actual_count}"
    )

assert Counter(
    item["sentence_count"] for item in EMBEDDED_TEST_DIARIES
) == Counter({2: 7, 3: 8, 4: 8, 5: 7})

assert Counter(
    item["target_emotion"] for item in EMBEDDED_TEST_DIARIES
) == Counter({
    "행복": 5, "기대": 5, "슬픔": 4, "분노": 4,
    "불안": 4, "피로": 4, "중립": 4
})

# 데모 데이터는 매 실행마다 기준 데이터로 재생성하여
# 이전 테스트/수정 내용이 남지 않도록 합니다.
demo_frame = pd.DataFrame([
    {"date": item["date"], "text": item["text"]}
    for item in EMBEDDED_TEST_DIARIES
])
demo_frame.to_csv(DEMO_DIARY_FILE, index=False, encoding="utf-8-sig")

pd.DataFrame(EMBEDDED_TEST_DIARIES).to_csv(
    DEMO_TARGET_FILE,
    index=False,
    encoding="utf-8-sig"
)

# 사용자 데이터는 별도 파일에 보존합니다.
if not USER_DIARY_FILE.exists():
    pd.DataFrame(columns=["date", "text"]).to_csv(
        USER_DIARY_FILE,
        index=False,
        encoding="utf-8-sig"
    )

# ---------------------------------------------------------
# 11 전용 결과 저장 함수
# 10의 diary_daily_context_v4.csv 등을 절대 덮어쓰지 않습니다.
# ---------------------------------------------------------
DAILY_EMPTY_COLUMNS = [
    "date", "text",
    "positive_score", "positive_emotion",
    "negative_score", "negative_emotion",
    "polarity_gap", "neutral_score",
    "raw_main_emotion", "main_emotion",
    "representative_score", "runner_up_emotion",
    "runner_up_score", "representative_gap",
    "dominance_level", "direct_emotions",
    "excluded_mentions", "context_adjusted_labels",
    "valence", "top_emotions", "comment"
]

def build_daily_outputs():
    day_rows = []
    sentence_rows = []

    for _, row in diary.iterrows():
        day_result, sentence_result, _ = analyze_text(row["text"])
        day_rows.append({"date": row["date"], "text": row["text"], **day_result})
        for sentence in sentence_result:
            sentence_rows.append({"date": row["date"], **sentence})

    if day_rows:
        daily_frame = pd.DataFrame(day_rows)
        daily_frame["comment"] = daily_frame.apply(daily_comment, axis=1)
    else:
        daily_frame = pd.DataFrame(columns=DAILY_EMPTY_COLUMNS)

    sentence_frame = pd.DataFrame(sentence_rows)

    tag = "demo_2026_09" if CURRENT_DATA_MODE == "demo" else "user"
    daily_frame.to_csv(
        output_dir / f"diary_daily_11_{tag}.csv",
        index=False,
        encoding="utf-8-sig"
    )
    sentence_frame.to_csv(
        output_dir / f"diary_sentences_11_{tag}.csv",
        index=False,
        encoding="utf-8-sig"
    )
    return daily_frame, sentence_frame


def _load_data_mode(mode):
    """demo/user 데이터셋을 서로 섞지 않고 전환한다."""
    global CURRENT_DATA_MODE, DIARY_FILE, diary, daily, sentences

    if mode not in {"demo", "user"}:
        raise ValueError("mode는 'demo' 또는 'user'여야 합니다.")

    CURRENT_DATA_MODE = mode
    DIARY_FILE = DEMO_DIARY_FILE if mode == "demo" else USER_DIARY_FILE

    diary = pd.read_csv(DIARY_FILE, encoding="utf-8-sig")

    if diary.empty:
        diary = pd.DataFrame(columns=["date", "text"])
    else:
        diary["date"] = pd.to_datetime(diary["date"]).dt.date
        diary = (
            diary.sort_values("date")
            .drop_duplicates("date", keep="last")
            .reset_index(drop=True)
        )

    with capture_output():
        daily, sentences = build_daily_outputs()

    return diary


# 최초에는 데모 데이터 30개를 한 번만 분석합니다.
_load_data_mode(CURRENT_DATA_MODE)

# 10의 show_month() 기본 인자 호환용 값.
# daily가 생성된 뒤에만 계산하므로 NameError가 발생하지 않습니다.
if len(diary):
    _default_date = max(diary["date"])
else:
    _default_date = date.today()

SELECT_YEAR = _default_date.year
SELECT_MONTH = _default_date.month


# ---------------------------------------------------------
# 11용 안전한 월간 데이터 선택
# 내 일기가 0개여도 열 구조를 유지한 빈 DataFrame을 반환합니다.
# ---------------------------------------------------------
def get_month_data(year, month):
    if daily is None or "date" not in daily.columns or daily.empty:
        return daily.iloc[0:0].copy()

    mask = daily["date"].apply(
        lambda day: (
            pd.notna(day)
            and day.year == int(year)
            and day.month == int(month)
        )
    )

    return (
        daily.loc[mask]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
    )


TEST_TARGETS = pd.read_csv(
    DEMO_TARGET_FILE,
    encoding="utf-8-sig"
)

print(
    "준비가 완료되었습니다. "
    "데모 모드에는 2026년 9월 예시 일기 30개가 등록되어 있습니다."
)


감정 분석 시스템을 준비하고 있습니다.
처음 실행하거나 calibration 캐시가 없으면 준비 시간이 더 걸릴 수 있습니다.
준비가 완료되었습니다. 데모 모드에는 2026년 9월 예시 일기 30개가 등록되어 있습니다.


## 사용자 화면

- **데모 데이터 30개**는 2026년 9월 1~30일에 한 편씩 배치되어 있습니다.
- 데모 데이터는 실행할 때마다 기준 상태로 복원되며 저장할 수 없습니다.
- 실제 일기를 저장하려면 상단에서 **내 일기** 모드로 전환합니다.
- 월간 감정은 연도와 월 기준으로만 조회하며 주차 선택 기능은 없습니다.


In [2]:
# 2. 사용자용 버튼 인터페이스
from datetime import date
from IPython.display import display, HTML, clear_output


# ---------------------------------------------------------
# 사용자용 하루 카드: 작성한 일기 원문까지 함께 표시
# ---------------------------------------------------------
USER_CARD_EXTRA_CSS = """
<style>
.dcard .diary-original{
    margin-top:8px;
    padding:12px 14px;
    background:#fafafa;
    border:1px solid #ececec;
    border-radius:10px;
    white-space:normal;
    font-size:14px;
    line-height:1.7;
}
.user-help{
    font-family:'Malgun Gothic','Apple SD Gothic Neo','Noto Sans KR',sans-serif;
    color:#666;
    font-size:13px;
    line-height:1.6;
}
.user-status-ok{color:#287a4b;font-weight:700}
.user-status-error{color:#b23b3b;font-weight:700}
</style>
"""


def render_user_day_card(text, result, day=None):
    """작성한 일기 원문 + 사용자용 감정 분석 결과."""
    day = day or date.today()
    if isinstance(day, str):
        day = pd.to_datetime(day).date()

    main = result["main_emotion"]
    color = EMOTION_COLORS[main]

    tops = [
        item.strip()
        for item in str(result["top_emotions"]).split(",")
        if item.strip()
    ][:5]

    chips = "".join(
        f'<span class="chip" '
        f'style="background:{_soft(_label_color(item))}">'
        f'{_esc(item)}</span>'
        for item in tops
    ) or '<span class="sub">뚜렷한 세부 감정이 없습니다.</span>'

    diary_html = _esc(text).replace("\n", "<br>")

    return CARD_CSS + USER_CARD_EXTRA_CSS + f"""
<div class="dcard">
  <h3>{day.year}년 {day.month}월 {day.day}일 ({WEEKDAYS[day.weekday()]}) 일기</h3>

  <div class="sec">작성한 일기</div>
  <div class="diary-original">{diary_html}</div>

  <div class="sec">대표 감정</div>
  <div class="main">
    <span class="chip-main"
          style="background:{_soft(color, '66')}">{_esc(main)}</span>
    <span class="sub">{_esc(dominance_text(result))}</span>
  </div>

  <div class="sec">긍정 · 부정</div>
  {_bar("긍정", result["positive_score"], "#4CAF7A")}
  {_bar("부정", result["negative_score"], "#E06C66")}

  <div class="sec">오늘 나타난 감정</div>
  <div class="chips">{chips}</div>

  <div class="comment">{_esc(user_comment(result))}</div>
</div>
"""


def analyze_for_user(text, day=None):
    """기존 10의 NLP 분석 로직을 그대로 사용해 사용자 카드에 필요한 값을 반환."""
    day = day or date.today()
    result, sentence_rows, _ = analyze_text(text)
    return result, sentence_rows, day


# ---------------------------------------------------------
# 공통 UI 스타일
# ---------------------------------------------------------
full_width = widgets.Layout(width="100%")
button_layout = widgets.Layout(width="145px", height="38px")
section_layout = widgets.Layout(
    width="760px",
    max_width="100%",
    padding="8px 4px 14px 4px"
)

header = widgets.HTML(
    value="""
    <div style="
        font-family:'Malgun Gothic','Apple SD Gothic Neo','Noto Sans KR',sans-serif;
        max-width:760px;padding:18px 20px;margin-bottom:10px;
        border:1px solid #e5e5e5;border-radius:14px;background:#fff;">
      <div style="font-size:23px;font-weight:700;">한국어 일기 감정 분석 시스템</div>
      <div style="margin-top:7px;color:#666;font-size:13px;">
        일기를 작성하면 KoBERT 기반 다중 감정 분석과 문맥 검증 결과를 보여줍니다.
      </div>
    </div>
    """
)

mode_selector = widgets.ToggleButtons(
    options=[
        ("데모 데이터 30개", "demo"),
        ("내 일기", "user"),
    ],
    value=CURRENT_DATA_MODE,
    description="데이터",
    button_style="",
    layout=widgets.Layout(width="430px")
)

system_status = widgets.HTML(value="")


def _mode_status_html():
    if CURRENT_DATA_MODE == "demo":
        return (
            '<span class="user-status-ok">● 데모 모드</span><br>'
            '<span style="color:#666;font-size:12px;">'
            '2026년 9월 예시 일기 30개를 사용합니다. '
            '기준 데이터를 보호하기 위해 저장 기능은 비활성화됩니다.'
            '</span>'
        )
    count = len(diary)
    return (
        '<span class="user-status-ok">● 내 일기 모드</span><br>'
        '<span style="color:#666;font-size:12px;">'
        f'직접 저장한 일기 {count}개를 사용합니다. '
        '데모 데이터와는 별도 파일에 저장됩니다.'
        '</span>'
    )


# =========================================================
# TAB 1. 일기 작성
# =========================================================
diary_help = widgets.HTML(
    value="""
    <div class="user-help">
      <b>날짜를 선택한 뒤 그날의 일기를 자유롭게 작성해주세요.</b><br>
      기본 날짜는 오늘이며, 과거 날짜도 직접 선택할 수 있습니다.
      여러 줄 입력이 가능하고 코드를 수정할 필요가 없습니다.
    </div>
    """
)

diary_date = widgets.DatePicker(
    description="작성 날짜",
    value=date.today(),
    layout=widgets.Layout(width="250px", margin="6px 0 2px 0")
)

diary_input = widgets.Textarea(
    value="",
    placeholder=(
        "예) 오늘 시험을 잘 봐서 뿌듯했다.\n"
        "다음 발표가 걱정되고 조금 피곤했다."
    ),
    description="",
    layout=widgets.Layout(
        width="100%",
        height="180px",
        margin="8px 0 10px 0"
    )
)

analyze_button = widgets.Button(
    description="감정 분석",
    button_style="primary",
    icon="search",
    tooltip="저장하지 않고 감정 분석 결과만 확인합니다.",
    layout=button_layout
)

save_button = widgets.Button(
    description="분석 후 저장",
    button_style="success",
    icon="save",
    tooltip="선택한 날짜로 분석 결과와 일기를 저장합니다.",
    layout=button_layout
)

clear_button = widgets.Button(
    description="입력 지우기",
    button_style="",
    icon="trash",
    tooltip="현재 입력한 일기를 지웁니다.",
    layout=button_layout
)

diary_status = widgets.HTML(value="")
diary_output = widgets.Output(
    layout=widgets.Layout(width="100%", margin="8px 0 0 0")
)


# =========================================================
# TAB 2. 월간 감정
# =========================================================
today = date.today()

month_year = widgets.Dropdown(
    options=[today.year],
    value=today.year,
    description="연도",
    layout=widgets.Layout(width="180px")
)

month_month = widgets.Dropdown(
    options=[(f"{m}월", m) for m in range(1, 13)],
    value=today.month,
    description="월",
    layout=widgets.Layout(width="160px")
)

month_button = widgets.Button(
    description="월간 감정 보기",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="170px", height="38px")
)

month_status = widgets.HTML(value="")
month_output = widgets.Output(
    layout=widgets.Layout(width="100%", margin="8px 0 0 0")
)

month_help = widgets.HTML(
    value="""
    <div class="user-help">
      <b>확인할 연도와 월만 선택하세요.</b><br>
      월간 감정 요약, 감정 달력, 일별 긍정 그래프와 부정 그래프를 한 번에 보여줍니다.
    </div>
    """
)


# =========================================================
# TAB 3. 날짜별 조회
# =========================================================
date_help = widgets.HTML(
    value="""
    <div class="user-help">
      <b>저장한 일기를 날짜별로 다시 확인할 수 있습니다.</b><br>
      날짜를 선택하면 작성했던 원문과 현재 분석 결과를 함께 보여줍니다.
    </div>
    """
)

date_picker = widgets.DatePicker(
    description="날짜",
    value=today,
    layout=widgets.Layout(width="250px")
)

date_button = widgets.Button(
    description="일기 보기",
    button_style="primary",
    icon="calendar",
    layout=widgets.Layout(width="140px", height="38px")
)

date_status = widgets.HTML(value="")
date_output = widgets.Output(
    layout=widgets.Layout(width="100%", margin="8px 0 0 0")
)


def _refresh_date_controls(preferred_day=None):
    """현재 데이터셋에 맞춰 연도/월/날짜 선택기를 갱신한다."""
    if len(diary):
        default_day = preferred_day or max(diary["date"])
        years = sorted({d.year for d in diary["date"]} | {today.year}, reverse=True)
    else:
        default_day = preferred_day or today
        years = [today.year]

    month_year.options = years
    month_year.value = default_day.year if default_day.year in years else years[0]
    month_month.value = default_day.month
    date_picker.value = default_day


def _set_diary_buttons_disabled(disabled):
    analyze_button.disabled = disabled
    clear_button.disabled = disabled
    # 데모 모드에서는 기준 데이터 보호를 위해 저장 버튼 비활성화
    save_button.disabled = disabled or CURRENT_DATA_MODE == "demo"


def _clear_all_outputs():
    diary_status.value = ""
    month_status.value = ""
    date_status.value = ""
    with diary_output:
        clear_output()
    with month_output:
        clear_output()
    with date_output:
        clear_output()


def _update_mode_ui(preferred_day=None):
    system_status.value = _mode_status_html()
    save_button.disabled = CURRENT_DATA_MODE == "demo"
    _refresh_date_controls(preferred_day=preferred_day)

    if CURRENT_DATA_MODE == "demo":
        diary_help.value = """
        <div class="user-help">
          <b>데모 모드에서는 감정 분석을 자유롭게 시험할 수 있습니다.</b><br>
          예시 30개 기준 데이터를 보호하기 위해 저장은 비활성화되어 있습니다.
          저장하려면 위에서 <b>내 일기</b> 모드로 전환해주세요.
        </div>
        """
    else:
        diary_help.value = """
        <div class="user-help">
          <b>날짜를 선택한 뒤 그날의 일기를 자유롭게 작성해주세요.</b><br>
          여러 줄 입력이 가능하며, 선택한 날짜로 저장됩니다.
        </div>
        """


def _on_mode_changed(change):
    if change["name"] != "value" or change["new"] == change["old"]:
        return

    mode_selector.disabled = True
    system_status.value = "데이터 모드를 전환하고 있습니다..."

    try:
        _load_data_mode(change["new"])
        _clear_all_outputs()
        _update_mode_ui()
    except Exception as exc:
        system_status.value = (
            '<span class="user-status-error">'
            f"모드 전환 중 문제가 발생했습니다: {_esc(exc)}"
            "</span>"
        )
    finally:
        mode_selector.disabled = False


def _handle_diary(save=False):
    text = diary_input.value.strip()
    selected_day = diary_date.value

    if selected_day is None:
        diary_status.value = (
            '<span class="user-status-error">'
            "작성 날짜를 먼저 선택해주세요."
            "</span>"
        )
        return

    if not text:
        diary_status.value = (
            '<span class="user-status-error">'
            "일기 내용을 먼저 입력해주세요."
            "</span>"
        )
        return

    if save and CURRENT_DATA_MODE == "demo":
        diary_status.value = (
            '<span class="user-status-error">'
            "데모 데이터에는 저장할 수 없습니다. "
            "상단에서 '내 일기' 모드로 전환해주세요."
            "</span>"
        )
        return

    _set_diary_buttons_disabled(True)
    diary_date.disabled = True
    diary_status.value = "감정을 분석하고 있습니다..."

    try:
        result, _, day = analyze_for_user(text, selected_day)
        existed = bool((diary["date"] == day).any())

        if save:
            _save_diary_quiet(text, day)
            _refresh_date_controls(preferred_day=day)

        with diary_output:
            clear_output(wait=True)
            display(HTML(render_user_day_card(text, result, day)))

            if save:
                save_message = (
                    f"{day}의 기존 일기를 새 내용으로 업데이트했습니다."
                    if existed
                    else f"{day} 일기를 저장했습니다."
                )
                display(HTML(
                    f'<div style="max-width:620px;margin:5px 0 15px 0;'
                    f'color:#287a4b;font-weight:700;">'
                    f'{save_message} '
                    f'월간 감정 기록에 반영되었습니다.</div>'
                ))

        diary_status.value = (
            '<span class="user-status-ok">분석이 완료되었습니다.</span>'
        )
        system_status.value = _mode_status_html()

    except Exception as exc:
        diary_status.value = (
            '<span class="user-status-error">'
            f"분석 중 문제가 발생했습니다: {_esc(exc)}"
            "</span>"
        )
    finally:
        diary_date.disabled = False
        _set_diary_buttons_disabled(False)


def _on_analyze_clicked(_):
    _handle_diary(save=False)


def _on_save_clicked(_):
    _handle_diary(save=True)


def _on_clear_clicked(_):
    diary_input.value = ""
    diary_date.value = date.today()
    diary_status.value = ""
    with diary_output:
        clear_output()


def _on_month_clicked(_):
    year = int(month_year.value)
    month = int(month_month.value)

    if daily is None or daily.empty:
        month_status.value = (
            '<span class="user-status-error">'
            "아직 저장된 일기가 없습니다. "
            "'일기 작성' 탭에서 일기를 먼저 저장해주세요."
            "</span>"
        )
        with month_output:
            clear_output()
        return

    selected = get_month_data(year, month)

    if selected.empty:
        month_status.value = (
            '<span class="user-status-error">'
            f"{year}년 {month}월에 저장된 일기가 없습니다."
            "</span>"
        )
        with month_output:
            clear_output()
        return

    month_button.disabled = True
    month_status.value = "월간 감정 기록을 불러오고 있습니다..."

    try:
        with month_output:
            clear_output(wait=True)

            display(HTML(render_month_card(year, month)))

            display(HTML(
                '<div style="font-weight:700;margin:18px 0 4px 0;">'
                "감정 달력</div>"
            ))
            plot_emotion_calendar(year, month)

            display(HTML(
                '<div style="font-weight:700;margin:18px 0 4px 0;">'
                "긍정 감정 변화</div>"
            ))
            plot_positive_graph(year, month)

            display(HTML(
                '<div style="font-weight:700;margin:18px 0 4px 0;">'
                "부정 감정 변화</div>"
            ))
            plot_negative_graph(year, month)

        month_status.value = (
            '<span class="user-status-ok">'
            f"{year}년 {month}월 감정 기록입니다."
            "</span>"
        )

    except Exception as exc:
        month_status.value = (
            '<span class="user-status-error">'
            f"월간 기록을 불러오는 중 문제가 발생했습니다: {_esc(exc)}"
            "</span>"
        )
    finally:
        month_button.disabled = False


def _on_date_clicked(_):
    selected_day = date_picker.value

    if selected_day is None:
        date_status.value = (
            '<span class="user-status-error">'
            "확인할 날짜를 선택해주세요."
            "</span>"
        )
        return

    selected = diary[diary["date"] == selected_day]

    if selected.empty:
        date_status.value = (
            '<span class="user-status-error">'
            f"{selected_day}에 저장된 일기가 없습니다."
            "</span>"
        )
        with date_output:
            clear_output()
        return

    date_button.disabled = True
    date_status.value = "저장된 일기를 불러오고 있습니다..."

    try:
        text = str(selected.iloc[-1]["text"])
        result, _, _ = analyze_for_user(text, selected_day)

        with date_output:
            clear_output(wait=True)
            display(HTML(
                render_user_day_card(
                    text,
                    result,
                    selected_day
                )
            ))

        date_status.value = (
            '<span class="user-status-ok">'
            f"{selected_day} 일기를 불러왔습니다."
            "</span>"
        )

    except Exception as exc:
        date_status.value = (
            '<span class="user-status-error">'
            f"일기를 불러오는 중 문제가 발생했습니다: {_esc(exc)}"
            "</span>"
        )
    finally:
        date_button.disabled = False


analyze_button.on_click(_on_analyze_clicked)
save_button.on_click(_on_save_clicked)
clear_button.on_click(_on_clear_clicked)
month_button.on_click(_on_month_clicked)
date_button.on_click(_on_date_clicked)
mode_selector.observe(_on_mode_changed, names="value")


diary_tab = widgets.VBox(
    [
        diary_help,
        diary_date,
        diary_input,
        widgets.HBox(
            [analyze_button, save_button, clear_button],
            layout=widgets.Layout(gap="8px", flex_flow="row wrap")
        ),
        diary_status,
        diary_output,
    ],
    layout=section_layout
)

month_tab = widgets.VBox(
    [
        month_help,
        widgets.HBox(
            [month_year, month_month, month_button],
            layout=widgets.Layout(gap="8px", flex_flow="row wrap")
        ),
        month_status,
        month_output,
    ],
    layout=section_layout
)

date_tab = widgets.VBox(
    [
        date_help,
        widgets.HBox(
            [date_picker, date_button],
            layout=widgets.Layout(gap="8px", flex_flow="row wrap")
        ),
        date_status,
        date_output,
    ],
    layout=section_layout
)

tabs = widgets.Tab(
    children=[diary_tab, month_tab, date_tab],
    layout=widgets.Layout(width="780px", max_width="100%")
)
tabs.set_title(0, "일기 작성")
tabs.set_title(1, "월간 감정")
tabs.set_title(2, "날짜별 조회")

_update_mode_ui()

display(
    widgets.VBox(
        [
            header,
            mode_selector,
            system_status,
            tabs,
        ],
        layout=widgets.Layout(width="800px", max_width="100%")
    )
)
